# 第一轮：流式频带序列记忆与声学回传
## 操作步骤
1. 从 [GitHub 项目](https://github.com/DoctorAwe/Parrot) 在 Google Colab 打开 `trial1_colab.ipynb`。若最近修改尚未推送，先推送到 GitHub；Colab 只会获取仓库里已经提交的版本。
2. 在菜单 **运行时 → 更改运行时类型** 中选择 GPU（若没有可用 GPU，可用 CPU，但会慢）。
3. 运行下面的仓库获取单元格，脚本从 GitHub 克隆到 `/content/Parrot`；然后挂载 Google Drive。无需手动上传代码。
4. 运行快速检查单元格，确认没有异常且生成 `trial1_smoke/metrics.json`。它只检查程序能跑通，不能证明模型学会了任务。
5. 运行正式试训单元格，等待训练和三组测试完成。不要在训练中重启运行时。
6. 运行结果展示单元格，查看准确率并试听输入、目标输出、模型输出。
7. 运行最后的打包单元格。结果保存在 Google Drive 的 `MyDrive/Parrot/trial1_results/`，压缩包保存在 `MyDrive/Parrot/trial1_results.zip`。把压缩包返回给我；如果不方便，至少返回 `metrics.json` 和训练末尾日志。

这是阶段 A 的可控频带信号试验，不是自然语言语音模型。当前阶段不读取声音样本；后续真实语音样本统一从 `MyDrive/Parrot/audio/` 读取。模型没有离散断句状态、文本中间表示或内部输出副本。所有训练与测试都在本 Colab 运行。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
assert (repo / 'trial1.py').is_file(), 'GitHub 仓库中缺少 trial1.py；请检查推送状态'
print('代码版本:', subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip())
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
drive_root = Path('/content/drive/MyDrive/Parrot')
audio_dir = drive_root / 'audio'  # 下一阶段真实语音样本目录；本轮不读取
result_dir = drive_root / 'trial1_results'
drive_root.mkdir(parents=True, exist_ok=True)
print('样本目录:', audio_dir)
print('结果目录:', result_dir)


先运行快速检查。它只验证代码可以完成前向、反向和结果导出，不代表模型已学会复述。


In [ ]:
!python trial1.py --smoke --batch 4 --out /content/Parrot/trial1_smoke


正式试训。建议先在 Colab 菜单中选择 GPU；没有 GPU 也可运行，但可能很慢。结果包含无回传与两种声学回传条件下的序列准确率。指标低说明第一版架构或训练任务仍需修改，不要将它解释为自然语音复述能力。


In [ ]:
!python trial1.py --steps 600 --batch 32 --eval 100 --out {result_dir}


In [ ]:
import json
from pathlib import Path
from IPython.display import Audio, display
print(json.dumps(json.loads((result_dir / 'metrics.json').read_text()), ensure_ascii=False, indent=2))
for filename in ('heard_input.wav', 'target_output.wav', 'model_output.wav'):
    print(filename)
    display(Audio(filename=str(result_dir / filename)))


### 导出与回传
压缩包包含模型权重、指标和三段试听音频，保存在 Google Drive 的 `MyDrive/Parrot/` 中。把压缩包返回给我，我会根据实际结果判断是否需要修改记忆结构、训练目标或数据生成方式。


In [ ]:
import shutil
archive = shutil.make_archive(str(drive_root / 'trial1_results'), 'zip', root_dir=str(drive_root), base_dir='trial1_results')
print('结果压缩包:', archive)
